# RKPCN Single-Replicate Analysis

Framework for diagnosing RKPCN behavior across different settings.

**Workflow**: Configure → Reconstruct → Run variants → Inspect output.

Helper functions in `rkpcn_analysis/` handle reconstruction, running, diagnostics, and plotting.

In [ ]:
from jax import config
config.update('jax_enable_x64', True)

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt

from rkpcn_analysis.reconstruct import reconstruct_replicate, load_saved_data
from rkpcn_analysis.runners import (
    run_rkpcn_variant, run_rkpcn_sweep, run_rkpcn_adaptive,
    get_adapted_proposal, build_log_density,
)
from rkpcn_analysis.diagnostics import summary_table, w2_table, autocorrelation
from rkpcn_analysis.plots import (
    plot_gp_predictive, plot_density_heatmaps,
    plot_traces, plot_acf, plot_samples_vs_ep,
    plot_adaptation_history,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12

## 1. Configuration

In [ ]:
from pathlib import Path

# --- Point this at your experiment output ---
EXPERIMENT_NAME = 'vsem_local_test'   # 'vsem' for full run
SETUP_NAME = 'clip_gp_N4'            # e.g., 'gp_N4', 'clip_gp_N8'
REP_IDX = 0

repo_root = Path('.').resolve().parents[1]
base_dir = repo_root / 'out' / EXPERIMENT_NAME

print(f'Experiment: {EXPERIMENT_NAME}')
print(f'Setup: {SETUP_NAME}, rep: {REP_IDX}')
print(f'Base dir: {base_dir}')

## 2. Reconstruct replicate and load saved data

In [ ]:
rep, key_run = reconstruct_replicate(base_dir, SETUP_NAME, REP_IDX)
saved = load_saved_data(base_dir, SETUP_NAME, REP_IDX)

par_names = list(rep.grid.dim_names)
gd = saved['grid_densities']

print(f'Parameters: {par_names}')
if saved['samples'] is not None:
    print(f'Sample keys: {list(saved["samples"].keys())}')

## 3. GP surrogate visualization

Predictive mean, standard deviation, and coefficient of variation.
Design points shown in red/blue.

In [ ]:
fig, _ = plot_gp_predictive(rep)
plt.show()

## 4. Grid-based density heatmaps

Exact posterior, plug-in mean, EUP, and EP densities on the grid.

In [ ]:
if gd is not None:
    fig, _ = plot_density_heatmaps(gd, rep.grid,
                                    names=['exact', 'mean', 'eup', 'ep'])
    plt.show()
else:
    print('No grid densities saved — run the experiment first.')

## 5. Get adapted proposal covariance

Run a short exact-posterior MCMC to get a well-tuned u-proposal.

In [ ]:
prop_cov, exact_samp = get_adapted_proposal(jr.key(42), rep)
print('Adapted proposal covariance:')
print(np.array(prop_cov))
print(f'\nExact posterior sample std: {np.array(exact_samp.std(0))}')

## 6. Baseline RKPCN sweep

All four $\rho$ values with standard settings and adapted proposal.

In [ ]:
baseline_variants = [
    {'rho': 0.0,  'label': 'cut'},
    {'rho': 0.9,  'label': 'rkpcn90'},
    {'rho': 0.95, 'label': 'rkpcn95'},
    {'rho': 0.99, 'label': 'rkpcn99'},
]

baseline_results = run_rkpcn_sweep(
    key=jr.key(100), rep=rep,
    variants=baseline_variants,
    prop_cov=prop_cov,
    n_total=55_000, n_burnin=50_000,
)

In [ ]:
_ = summary_table(baseline_results, par_names=par_names)

## 7. Trace plots and ACF

In [ ]:
trace_figs = plot_traces(baseline_results, par_names=par_names)
for fig in trace_figs:
    plt.show()

In [ ]:
fig, _ = plot_acf(baseline_results, par_names=par_names, max_lag=3000)
plt.show()

## 8. Samples vs EP density

Scatter plots overlaid on EP density contours.
Includes saved exact/mean/eup for reference.

In [ ]:
reference_samples = {}
if saved['samples'] is not None:
    for name in ['exact', 'mean', 'eup']:
        if name in saved['samples']:
            reference_samples[name] = np.array(saved['samples'][name])

fig, _ = plot_samples_vs_ep(
    baseline_results, ep_density=gd['ep'], grid=rep.grid,
    reference_samples=reference_samples, thin=5)
plt.show()

## 9. W2 to grid EP (KDE-on-grid approach)

In [ ]:
# Combine reference samples + RKPCN results for unified W2 table
w2_input = {}
for name, samp in reference_samples.items():
    w2_input[name] = {'post_burnin': samp, 'label': name}
w2_input.update(baseline_results)

w2_vals = w2_table(w2_input, ep_grid_density=gd['ep'], grid=rep.grid,
                   par_names=par_names)

---

## 10. Experiment: Proposal scale sweep ($\rho = 0.99$)

In [ ]:
scale_variants = []
for s in [0.1, 0.5, 1.0, 2.0, 5.0]:
    scale_variants.append({
        'rho': 0.99,
        'label': f'rho99_s{s}',
        'prop_cov': s**2 * prop_cov,
    })

scale_results = run_rkpcn_sweep(
    key=jr.key(200), rep=rep, variants=scale_variants,
    prop_cov=prop_cov, n_total=55_000, n_burnin=50_000)

_ = summary_table(scale_results, par_names=par_names)

In [ ]:
fig, _ = plot_samples_vs_ep(
    scale_results, ep_density=gd['ep'], grid=rep.grid, thin=5)
plt.show()

_ = w2_table(scale_results, ep_grid_density=gd['ep'], grid=rep.grid)

## 11. Experiment: Spherical vs adapted proposal ($\rho = 0.99$)

In [ ]:
d = rep.posterior.dim

sphere_variants = [
    {'rho': 0.99, 'label': 'adapted', 'prop_cov': prop_cov},
    {'rho': 0.99, 'label': 'sphere_0.01', 'prop_cov': 0.01 * jnp.eye(d)},
    {'rho': 0.99, 'label': 'sphere_0.1', 'prop_cov': 0.1 * jnp.eye(d)},
    {'rho': 0.99, 'label': 'sphere_1.0', 'prop_cov': 1.0 * jnp.eye(d)},
]

sphere_results = run_rkpcn_sweep(
    key=jr.key(300), rep=rep, variants=sphere_variants,
    prop_cov=prop_cov, n_total=55_000, n_burnin=50_000)

_ = summary_table(sphere_results, par_names=par_names)

In [ ]:
fig, _ = plot_samples_vs_ep(
    sphere_results, ep_density=gd['ep'], grid=rep.grid, thin=5)
plt.show()

_ = w2_table(sphere_results, ep_grid_density=gd['ep'], grid=rep.grid)

## 12. Experiment: Scaled iterations for high $\rho$

In [ ]:
iter_variants = [
    {'rho': 0.0,  'label': 'cut_std',   'n_total': 55_000, 'n_burnin': 50_000},
    {'rho': 0.99, 'label': 'rho99_std', 'n_total': 55_000, 'n_burnin': 50_000},
    {'rho': 0.99, 'label': 'rho99_5x',  'n_total': 275_000, 'n_burnin': 250_000},
    {'rho': 0.99, 'label': 'rho99_20x', 'n_total': 1_100_000, 'n_burnin': 1_000_000},
]

iter_results = run_rkpcn_sweep(
    key=jr.key(400), rep=rep, variants=iter_variants, prop_cov=prop_cov)

_ = summary_table(iter_results, par_names=par_names)

In [ ]:
fig, _ = plot_samples_vs_ep(
    iter_results, ep_density=gd['ep'], grid=rep.grid, thin=10)
plt.show()

_ = w2_table(iter_results, ep_grid_density=gd['ep'], grid=rep.grid, thin=10)

## 13. Experiment: EUP f-update (correlation pseudo-marginal)

Uses `_f_update_eup_cpm`: MH correction on f using the unnormalized
density ratio (targets EUP, not EP).

In [ ]:
from uncprop.core.samplers import _f_update_eup_cpm

eup_cpm_variants = [
    {'rho': 0.99, 'label': 'pcn_rho99',
     'f_update_fn': None},
    {'rho': 0.99, 'label': 'eup_cpm_rho99',
     'f_update_fn': _f_update_eup_cpm},
    {'rho': 0.95, 'label': 'eup_cpm_rho95',
     'f_update_fn': _f_update_eup_cpm},
    {'rho': 0.0,  'label': 'eup_cpm_rho0',
     'f_update_fn': _f_update_eup_cpm},
]

eup_cpm_results = run_rkpcn_sweep(
    key=jr.key(500), rep=rep, variants=eup_cpm_variants,
    prop_cov=prop_cov, n_total=55_000, n_burnin=50_000)

_ = summary_table(eup_cpm_results, par_names=par_names)

In [ ]:
fig, _ = plot_samples_vs_ep(
    eup_cpm_results, ep_density=gd['ep'], grid=rep.grid, thin=5)
plt.show()

_ = w2_table(eup_cpm_results, ep_grid_density=gd['ep'], grid=rep.grid)

## 14. Experiment: Adaptive proposal covariance

During burn-in, adapt $\Sigma_Q$ using Haario-style running covariance
plus Robbins-Monro scale adaptation targeting a specified acceptance rate.
After burn-in, the proposal is frozen and the post-burnin phase runs
via the fast JIT path.

**Key question**: does adapting to the *RKPCN implicit target* geometry
(rather than the exact posterior) produce a better proposal?

In [ ]:
# Adaptive proposal starting from the exact-posterior adapted covariance
adaptive_variants = [
    # Baseline: fixed proposal for comparison
    {"rho": 0.99, "label": "rho99_fixed"},
    # Adaptive: starts from same initial proposal, adapts during burnin
    {"rho": 0.99, "label": "rho99_adapt", "adaptive": True,
     "n_burnin": 50_000, "n_post": 5_000,
     "adapt_start": 500, "adapt_interval": 200, "target_accept": 0.30},
    # Adaptive with different target acceptance rate
    {"rho": 0.99, "label": "rho99_adapt_234", "adaptive": True,
     "n_burnin": 50_000, "n_post": 5_000,
     "adapt_start": 500, "adapt_interval": 200, "target_accept": 0.234},
    # Adaptive with spherical initial proposal
    {"rho": 0.99, "label": "rho99_adapt_sphere", "adaptive": True,
     "prop_cov_init": 0.1 * jnp.eye(rep.posterior.dim),
     "n_burnin": 50_000, "n_post": 5_000,
     "adapt_start": 500, "adapt_interval": 200, "target_accept": 0.30},
]

adaptive_results = run_rkpcn_sweep(
    key=jr.key(600), rep=rep,
    variants=adaptive_variants,
    prop_cov=prop_cov,
    n_total=55_000, n_burnin=50_000,
)

_ = summary_table(adaptive_results, par_names=par_names)

In [ ]:
# Adaptation history plots
for label, res in adaptive_results.items():
    if "adapt_history" in res:
        fig, _ = plot_adaptation_history(res)
        plt.show()
        print(f"Final proposal cov ({label}):")
        print(res["final_prop_cov"])
        print()

In [ ]:
# Compare samples against EP
fig, _ = plot_samples_vs_ep(
    adaptive_results, ep_density=gd["ep"], grid=rep.grid, thin=5)
plt.show()

_ = w2_table(adaptive_results, ep_grid_density=gd["ep"], grid=rep.grid)

In [ ]:
# Trace and ACF for adaptive variants
trace_figs = plot_traces(adaptive_results, par_names=par_names)
for fig in trace_figs:
    plt.show()

fig, _ = plot_acf(adaptive_results, par_names=par_names, max_lag=3000)
plt.show()

---

## 15. Custom experiment cell

Modify variants list and re-run to test new ideas quickly.

In [ ]:
custom_variants = [
    # Modify these as needed:
    {'rho': 0.99, 'label': 'custom_1', 'prop_cov': 2.0**2 * prop_cov,
     'n_total': 275_000, 'n_burnin': 250_000},
]

custom_results = run_rkpcn_sweep(
    key=jr.key(999), rep=rep, variants=custom_variants, prop_cov=prop_cov)

_ = summary_table(custom_results, par_names=par_names)

fig, _ = plot_samples_vs_ep(
    custom_results, ep_density=gd['ep'], grid=rep.grid, thin=10)
plt.show()

_ = w2_table(custom_results, ep_grid_density=gd['ep'], grid=rep.grid, thin=10)